In [6]:
def transform_sep_to_normalized_intervals_metadata(sep, variables_unique, normalized_intervals, d, boundaries):

    # Import all the necessary Python modules
    import numpy
    
    # Import "epstatKDTree" module
    import epstatKDTree
    
    # Extract dimension 'P' from 'sep.shape'
    P = len(sep.shape)
    
    # Extract number of terms 'J' in Fourier-series approximation from 'sep.shape'
    J = int((sep.shape[0] - 1) / 2)

    # Collect the number of intervals in 'normalized_intervals'
    num_intervals = normalized_intervals.shape[0]
    
    # Fetch the upper and the lower bounds for all relevant unique variables used in this construction
    lower = numpy.array([boundaries['lower'][variable] for variable in variables_unique])
    upper = numpy.array([boundaries['upper'][variable] for variable in variables_unique])
    
    # Initialize the array of cells and conditional medians
    new_normalized_intervals = numpy.empty((2*num_intervals, 2, P))
    conditional_medians = numpy.empty(I)
    
    # Run loop for each 'interval' in 'normalized_intervals'
    for i in range(num_intervals):

        # Fetch the i^th 'interval'
        interval = normalized_intervals[i]

        # Calculate the 'conditional_median' for the current 'interval'
        conditional_median_normalised = epstatKDTree.calculate_median_conditionl(sep, interval[0], interval[1], d)
        conditional_medians[i] = conditional_median_normalised * (upper[d] - lower[d]) + lower[d]

        # Construct interval corresponding to left child of 'normalized_intervals[i]'
        new_normalized_intervals[2*i] = interval
        new_normalized_intervals[2*i][1][d] = conditional_median_normalised

        # Construct cell corresponding to right child of 'normalized_cells[i]'
        new_normalized_intervals[2*i+1] = interval
        new_normalized_intervals[2*i+1][0][d] = conditional_median_normalised

    # Store 'new_normalized_intervals' and 'conditional_medians' in the dictionary 'normalized_intervals_metadata'
    normalized_intervals_metadata = {'normalized_intervals' : new_normalized_intervals, 'conditional_medians' : conditional_medians}
    
    return(normalized_intervals_metadata)

In [7]:
def build_tree_segment_from_sep(sep, variables_unique, indices_tuple, boundaries):

    # Import all the necessary Python modules
    import numpy
    
    # Import "epstatKDTree" module
    import epstatKDTree
    
    # Extract dimension 'P' from 'sep.shape'
    P = len(sep.shape)
    
    # Extract number of levels to build for the current 'tree_segment'
    D = len(indices_tuple)

    # Initialize the list of normalized intervals which represent cells for each levels of this 'tree_segment', we start with [(0, 0, . . ., 0), (1, 1, . . ., 1)]
    normalized_intervals = numpy.zeros((1, 2, P))
    normalized_intervals[0][1] = numpy.ones(P)
    
    # Initialize the list of conditional medians which represent conditional medians for each levels of this 'tree_segment'
    conditional_medians  = [None] * D
    
    # Initialize the list of variables along which, we split the cells for each levels of this 'tree_segment'
    splitting_variables = [None] * D
    
    # Iterate for each level of the current 'tree_segment' and build
    for d in range(D):
        
        # Identify the splitting axis along which current 'normalized_intervals' should be split
        splitting_axis = indices_tuple[d]
        
        # Split and extract information from the split
        normalized_intervals_metadata = epstatKDTree.transform_sep_to_normalized_cells_info(sep, variables_unique, normalized_cells, splitting_axis, boundaries)
        
        # Extract the new 'normalized_intervals'
        normalized_intervals = normalized_intervals_metadata['normalized_intervals']
        
        # Extract the new 'conditional_medians'
        conditional_medians[d] = normalized_cells_metadata['conditional_medians']
        
        # Extract splitting variable name
        splitting_variables[d] = variables_unique[splitting_axis]
        
    # Store 'Depth', 'Splitting variable' and 'Splitting points' for each levels in the dictionary 'tree'
    tree_segment = [{'Depth' : d, 'Splitting variable': splitting_variables[d], 'Splitting points': medians[d]} for d in range(D)]
    
    return(tree_segment)

In [8]:
def build_branch_metadata_from_sep(sep, variables_unique, indices_tuple, branch_index, tree_depth, branch_boundaries_list):

    # Import all the necessary Python modules
    import numpy
    
    # Import "epstatKDTree" module
    import epstatKDTree
   
    # Get the variable limits for current branch
    bounds = {'lower': branch_boundaries_list[branch_index]['lower'].copy(), 'upper': branch_boundaries_list[branch_index]['upper'].copy()}
    
    # Extract dimension 'P' from 'sep.shape'
    P = len(sep.shape)
    
    # Extract number of levels to build for the current 'branch_segment'
    D = len(indices_tuple)

    # Initialize the list of normalized cells which represent cells for each levels of this 'branch_segment', we start with [(0, 0, . . ., 0), (1, 1, . . ., 1)]
    normalized_intervals = numpy.zeros((1, 2, P))
    normalized_intervals[0][1] = numpy.ones(P)
    
    # Initialize the list of conditional medians which represent conditional medians for each levels of this 'branch_segment'
    conditional_medians  = [None] * D
    
    # Initialize the list of variables along which, we split the cells for each levels of this 'branch_segment'
    splitting_variables = [None] * D
    
    # Iterate for each level of the current 'branch_segment' and build
    for d in range(D):
        
        # Identify the splitting axis along which current 'normalized_intervals' should be split
        splitting_axis = indices_tuple[d]
        
        # Split and extract information from the split
        normalized_intervals_metadata = epstatKDTree.transform_sep_to_normalized_cells_info(sep, variables_unique, normalized_cells, splitting_axis, boundaries)
        
        # Extract the new 'normalized_intervals'
        normalized_intervals = normalized_intervals_metadata['normalized_intervals']
        
        # Extract the new 'conditional_medians'
        conditional_medians[d] = normalized_intervals_metadata['conditional_medians']
        
        # Extract splitting variable name
        splitting_variables[d] = variables_unique[splitting_axis]
        
    # Extract key-value pairs for current branch
    branch_info_key_value_pairs = [((d + tree_depth, splitting_variables[d]), [(branch_id, conditional_medians[d])]) for d in range(D)]

    return(branch_metadata_key_value_pairs)

In [9]:
def build_loacl_subtree_canonical(subset_data, loacl_subtree_depth, current_variable_index, variables):

    # Import necessary numpy functions
    from numpy import median
    
    # Get the cell-id of the 'subset_data'
    cell = data.loc[0, 'cell']
    subtree_bfs_list = [[]] * loacl_subtree_depth
    for d in range(local_depth):
        greater = 0
        tree[d] = []
        for k in range(leaf*2**d, (leaf+1)*2**d):
            medianCond = median(data.loc[data['leaf'] == k, variables[var_index]].values)
            tree[d].append(medianCond)
            greater += ((data['leaf'] == k) & (data[variables[var_index]] > medianCond)).values
        var_index = (var_index + 1) % len(variables)
        data.leaf = 2 * data.leaf + greater
    return tree


In [10]:
def localBranch(x, tree_depth, local_depth, start_var_index, variables):
    from numpy import array
    from pandas import DataFrame
    p = len(variables)
    data = DataFrame(x[1]).assign(leaf = x[0])
    subtree = getTree(data, local_depth, start_var_index, variables)
    branch = [((tree_depth + d, variables[(start_var_index + d) % p]), [(x[0], array(subtree[d]))]) for d in range(local_depth)]
    return(branch)

In [11]:
def lambda_tree_segment_sep_summand(row, variables_tree_segment, boundaries, J):

    # Import all the necessary Python modules
    import numpy
    
    # Get lower and upper limits of all the varibales in 'row'
    lower, upper = boundaries['lower'], boundaries['upper']
    
    # Normalize and store every variables in 'row', which are used to build this segment
    x = [(row[variable] - lower[variable]) / (upper[variable] - lower[variable]) for variable in variables_tree_segment]
    
    # Get dimension 'P' from 'x'
    P = len(x)
    
    # Get the number of elements 'K', in the parameter 'J'
    K = len(J)
    
    # Initialise list 'C' of length 'K', each element in 'C' correspond to an element in 'J'
    C = [None] * K
    
    # Initialise C[0] as an empty array of length 2 * J[0]
    C[0] = numpy.empty(2 * J[0])
    
    # Iterate for remaining elements of 'J'
    for k in range(1, K):
        
        # Initialise C[k] as an empty array of length J[k]
        C[k] = numpy.empty(J[k])
        
        # Set the first element as 1
        C[k][0] = 1
        
    # Initialise the summand 'CC' to 1     
    CC = 1
    
    # Iterate over each element 'xx' of the normalized row-vector 'x'
    for xx in x:
        
        # 'C[0]' is intended to be the array: '[cos(xx), sin(xx), cos^3(xx), sin(xx).cos^2(xx), cos^5(xx), sin(xx).cos^4(xx), ...]', and is calcualated via a for loop
        c = numpy.cos(xx)
        
        # 'm' stores the common multiplier 'cos^2(xx)'
        m = c*c
        
        # 'sin(xx) = sqrt(1 - cos^2(xx)).sign(xx)' for 'xx' in the interval: '(0, 1)', we skip the computationally expensive polynomial approximation of Sine function
        s = numpy.sqrt(1 - m) * numpy.sign(xx)
        
        # First two entries of 'C[0]' are 'cos(xx)' and 'sin(xx)'
        C[0][:2] = c, s
        
        # Iterate over '1 <= j < J[0]' in a for loop
        for j in range(1, J[0]):
            
            # Multiplication step with common multiplier 'm = cos^2(xx)'
            c *= m
            s *= m
            
            # '(2j+1)^th' and '(2j+2)^th' entries of 'C[0]' are 'cos^(2j+1)(xx)' and 'sin(xx).cos^(2j)(xx)'
            C[0][2*j: 2*j+2] = c, s
            
        # 'cc' is the tensor to track outer-products corresponding to the variable 'xx', which is initialised to 'C[0]'
        cc = C[0]
        
        # For '1 <= k < K', the common Cosine multiplier is 'cos(L_k * xx)' where 'L_k = 2 * J_1 * ... * J_{k-1}, so we initialize 'X = 2 * xx'
        X = 2*xx
        
        # Iterate over '1 <= k < K' in a for loop
        for k in range(1, K):
            
            # We use the relation 'L_k * xx = (L_{k-1} * xx) * J_{k-1}'
            X *= J[k-1]
            
            # 'C[k]' is intended to be the array: '[1, cos(L_k * xx), cos^2(L_k * xx), ...]', and is calcualated via a for loop
            c = 1
            
            # 'm' stores the common multiplier 'cos(L_k * xx)'
            m = numpy.cos(X)
            
            # Iterate over '1 <= j < J[k]' in a for loop
            for j in range(1, J[k]):
                
                # Multiplication step with common multiplier 'm = cos(L_k * xx)'
                c *= m
                
                # 'j^th' entry of 'C[k]' is 'cos^(j-1)(L_k * xx)'
                C[k][j] = c
                
            # Store the cumulative outer-product tensor corresponding to 'xx' in 'cc'    
            cc = numpy.outer(cc, C[k])
            
        # Flatten 'cc' to a 1D array and insert '1' at the start  
        cc = numpy.insert(cc.ravel(), 0, 1.)
        
        # Take cumulative outer product for ecah 'xx' in 'x' to get the final summand
        CC = numpy.outer(CC, cc)
        
    return(CC.ravel())

In [12]:
def lambda_modify_branch_index(row, tree, current_tree_height):

    # Import all the necessary Python modules
    import pyspark
    
    # Get the 'branch_index' of the 'row'
    branch_index = row['branch_index']
    
    # Traverse through the existing-tree from depth 'current_tree_height' to length of the tree'
    for d in range(height, len(tree)):
        
        # Check and go to left sub-tree 
        if row[tree[d]['Splitting variable']] <= tree[d]['Splitting points'][i]:
            
            # Change the current branch_index to that of the branch_index of the left child
            branch_index = 2*branch_index
        
        # Go to right sub-tree 
        else:
            
            # Change the current branch_index to that of the branch_index of the right child
            branch_index = 2*branch_index+1
            
    # Make a copy of the base dictionary of the row
    new_row_dictionary = row.asDict()
    
    # Adjust branch_index
    new_row_dictionary[branch_index] = branch_index

    # Convert the dictionary to a pyspark Row
    new_row = pyspark.sql.Row(**new_row_dictionary)
    
    return(new_row)

In [13]:
def lambda_branch_sep_summand(row, variables_branch_segment, branch_boundaries, J):

    # Import all the necessary Python modules
    import numpy
    
    # Import necessary numpy functions
    from numpy import empty, cos, sqrt, sign, outer, insert
    
    # Get the 'branch_index' of the 'row'
    branch_index = row['branch_index']
    
    # Get lower and upper limits of the varibales in 'row'
    lower, upper = branch_boundaries_list[i]['lower'].copy(), branch_boundaries_list[i]['upper'].copy()
    
    # Normalize and store every variables in 'row', which are used to build this segment
    x = [(row[variable] - lower[variable]) / (upper[variable] - lower[variable]) for variable in variables_branch_segment]
    
    # Get dimension 'P' from 'x'
    P = len(x)
    
    # Get the number of elements 'K', in the parameter 'J'
    K = len(J)
    
    # Initialise list 'C' of length 'K', each element in 'C' correspond to an element in 'J'
    C = [None] * K
    
    # Initialise C[0] as an empty array of length 2 * J[0]
    C[0] = numpy.empty(2 * J[0])
    
    # Iterate for remaining elements of 'J'
    for k in range(1, K):
        
        # Initialise C[k] as an empty array of length J[k]
        C[k] = numpy.empty(J[k])
        
        # Set the first element as 1
        C[k][0] = 1
        
    # Initialise the summand 'CC' to 1     
    CC = 1
    
    # Iterate over each element 'xx' of the normalized row-vector 'x'
    for xx in x:
        
        # 'C[0]' is intended to be the array: '[cos(xx), sin(xx), cos^3(xx), sin(xx).cos^2(xx), cos^5(xx), sin(xx).cos^4(xx), ...]', and is calcualated via a for loop
        c = numpy.cos(xx)
        
        # 'm' stores the common multiplier 'cos^2(xx)'
        m = c*c
        
        # 'sin(xx) = sqrt(1 - cos^2(xx)).sign(xx)' for 'xx' in the interval: '(0, 1)', we skip the computationally expensive polynomial approximation of Sine function
        s = numpy.sqrt(1 - m) * numpy.sign(xx)
        
        # First two entries of 'C[0]' are 'cos(xx)' and 'sin(xx)'
        C[0][:2] = c, s
        
        # Iterate over '1 <= j < J[0]' in a for loop
        for j in range(1, J[0]):
            
            # Multiplication step with common multiplier 'm = cos^2(xx)'
            c *= m
            s *= m
            
            # '(2j+1)^th' and '(2j+2)^th' entries of 'C[0]' are 'cos^(2j+1)(xx)' and 'sin(xx).cos^(2j)(xx)'
            C[0][2*j: 2*j+2] = c, s
            
        # 'cc' is the tensor to track outer-products corresponding to the variable 'xx', which is initialised to 'C[0]'
        cc = C[0]
        
        # For '1 <= k < K', the common Cosine multiplier is 'cos(L_k * xx)' where 'L_k = 2 * J_1 * ... * J_{k-1}, so we initialize 'X = 2 * xx'
        X = 2*xx
        
        # Iterate over '1 <= k < K' in a for loop
        for k in range(1, K):
            
            # We use the relation 'L_k * xx = (L_{k-1} * xx) * J_{k-1}'
            X *= J[k-1]
            
            # 'C[k]' is intended to be the array: '[1, cos(L_k * xx), cos^2(L_k * xx), ...]', and is calcualated via a for loop
            c = 1
            
            # 'm' stores the common multiplier 'cos(L_k * xx)'
            m = numpy.cos(X)
            
            # Iterate over '1 <= j < J[k]' in a for loop
            for j in range(1, J[k]):
                
                # Multiplication step with common multiplier 'm = cos(L_k * xx)'
                c *= m
                
                # 'j^th' entry of 'C[k]' is 'cos^(j-1)(L_k * xx)'
                C[k][j] = c
                
            # Store the cumulative outer-product tensor corresponding to 'xx' in 'cc'    
            cc = numpy.outer(cc, C[k])
            
        # Flatten 'cc' to a 1D array and insert '1' at the start  
        cc = numpy.insert(cc.ravel(), 0, 1.)
        
        # Take cumulative outer product for ecah 'xx' in 'x' to get the final summand
        CC = numpy.outer(CC, cc)

        # Form the key-value pair for branch sep summand
        branch_sep_summand_key_value_pair = (branch_index, CC.ravel())
        
    return(branch_sep_summand_key_value_pair)

In [14]:
def lambda_transform_branch_sep_to_branch_metadata(sep_key_value, tree_height, variables_unique, indices_tuple, branch_boundaries_list, J):

    # Import all the necessary Python modules
    import numpy
    
    # Import "epstatKDTree" module
    import epstatKDTree
   
    # Extract 'branch_index' key from 'sep_key_value_pair'
    branch_index_key = sep_key_value[0]
    
    # Normalize 'branch_index' key in a binary array
    branch_index_normalized = dict(reversed(list(enumerate(bin(branch_index_key)[2:].zfill(tree_height)))))
    
    # Convert the binary array to integer representing unique branch_index in a for loop
    branch_index = 0
    
    # Iterate over each level of the already constructed tree-segment
    for d in range(tree_height):
        
        # Check if current entry is 0   
        if branch_index_normalized[d] == '0':
            branch_index = 2 * branch_index
        
        # Check if current entry is 1       
        else:
            branch_index = 2 * branch_index + 1
            
    # Extract summation sep statistics from 'sep_key_value'
    sep = sep_key_value[1]
    
    # Standerdize the summation sep statistics
    sep_standardized = numpy.array(sep, dtype = float) / float(sep[0])
        
    # Transform the standerdize summation sep statistics to sep statistics which can be used to construct this branch-segment
    sep_branch = epstatKDTree.transform_PD_trigonometric_product_tensor_to_trigonometric_tensor(sep_standardized, J)
    
    # Construct branch of the KD-Tree and store the BFS representation meta-data of branch-levels in a branch metadata key-value pairs
    branch_metadata_key_value_pairs = epstatKDTree.build_branch_segment_from_sep(sep_branch, variables_unique, indices_tuple, branch_index, tree_height, branch_boundaries_list)
    
    return(branch_metadata_key_value_pairs)

In [ ]:

def lambda_merge_branches_in_local(row):
    rowDict = row.asDict()
    key = rowDict.pop('cell')
    value = [rowDict]
    return (key, value)


In [ ]:
def lambda_merge_branch_metadata(x):

    # Import all the necessary Python modules
    import numpy

    # Merge all values with common key in a single numpy array
    branch_metadata_key_value_pair = (x[0], numpy.array([dict(x[1])[i] for i in range(len(x[1]))]).ravel())
    
    return(branch_metadata_key_value_pair)

In [ ]:
def calculate_branch_boundaries_list(boundaries, tree):
    tree_depth = len(tree)
    branch_boundaries_list = [None] * (2**tree_depth)
    for branch_index in range(len(branch_boundaries_list)):
        lower = boundaries['lower'].copy()
        upper = boundaries['upper'].copy()
        branch_bin_id = dict(reversed(list(enumerate(bin(branch_index)[2:].zfill(tree_depth)))))
        i = 0
        for d in range(tree_depth):
            if branch_bin_id[d] == '0':
                upper[tree[d]['Splitting variable']] = tree[d]['Splitting points'][i]
                i = 2*i
            else:
                lower[tree[d]['Splitting variable']] = tree[d]['Splitting points'][i]
                i = 2*i+1
        branch_boundaries_list[branch_index] = {'lower' : lower, 'upper' : upper}
    return(branch_boundaries_list)

In [ ]:
def get_variable_indices(variables_tuple):
    
    # Collect unique set of variables in "variables_tuple"
    variables_unique = tuple(set(variables_tuple))
    
    # Obtain index of each variable in natural order
    indices_tuple = tuple([{variable: i for i, variable in enumerate(variables_unique)}[variable] for variable in variables_tuple])
    
    return(variables_unique, indices_tuple)

In [ ]:

def buildTree(data, boundaries, variables_tuple, J):
    from numpy import zeros
    from operator import add
    variables_unique, indices_tuple = getIndices(variables_tuple)
    sep_raw = data.rdd.map(lambda row: f_tree(row, variables_unique, boundaries, J)).reduce(add)
    wep_raw = zeros(len(sep_raw))
    wep_raw[0] = 1.
    for j in range(1, len(sep_raw)):
        wep_raw[j] = sep_raw[j] / sep_raw[0]
    wep = transform_PD_trigonometric_product_tensor_to_trigonometric_tensor(wep_raw, J)
    tree = wep2tree(wep, variables_unique, indices_tuple, boundaries)
    return(tree)


In [ ]:

def assignLeaf(data, tree, height):
    data = data.rdd.map(lambda row: f_assign_leaf(row, tree, height)).toDF()
    return data


In [ ]:

def addBranches(data, tree, branch_boundaries_list, variables_tuple, J):
    from operator import add
    variables_unique, indices_tuple = getIndices(variables_tuple)
    tree_depth = len(tree)
    sep_rdd = data.rdd.map(lambda row: f_branch_sep(row, variables_unique, branch_boundaries_list, J)).reduceByKey(add)
    branches_list = sep_rdd.flatMap(lambda x: f_branch_compute(x, tree_depth, variables_unique, indices_tuple, branch_boundaries_list, J)).reduceByKey(lambda x, y: x+y).map(lambda x: f_branch_merge(x)).collect()
    branches_dict = {r[0][0]: {'Depth' : r[0][0], 'Splitting variable': r[0][1], 'Splitting points': r[1]} for r in branches_list}
    for d in range(len(branches_dict)):
        tree.append(branches_dict[d+tree_depth])
    return(tree)


In [ ]:

def addLocalBranches(data, tree, local_depth, variables):
    tree_depth = len(tree)
    start_var_index = height % len(variables)
    branches_list = data.rdd.map(lambda x: f_branch_merge_local(x)).reduceByKey(lambda x, y: x+y).flatMap(lambda x: localBranch(x, tree_depth, local_depth, start_var_index, variables)).reduceByKey(lambda x, y: x+y).map(lambda x: f_branch_merge(x)).collect()
    branches_dict = {r[0][0]: {'Depth' : r[0][0], 'Splitting variable': r[0][1], 'Splitting points': r[1]} for r in branches_list}
    for d in range(len(branches_dict)):
        tree.append(branches_dict[d+tree_depth])
    return(tree)


In [ ]:

def bfs2dfs(index, depth, variables, flat_medians, parent=None):
    if index >= len(flat_medians) or flat_medians[index] is None:
        return None
    axis = depth % len(variables)
    leaf = index - (2 ** depth - 1)
    node = KDNode(
        median=flat_medians[index],
        variables = variables,
        axis=axis,
        level=depth,
        leaf=leaf,
        parent=parent
    )
    node.left = bfs2dfs(2 * index + 1, depth + 1, variables, flat_medians, parent=node)
    node.right = bfs2dfs(2 * index + 2, depth + 1, variables, flat_medians, parent=node)
    return node
